# Project Management Dashboard
This is a description of the Project Management Dashboard system, which provides a visual overview of dependencies and current releases across multiple subprojects in an organization.

**Note:** Before running this notebook, you must first run the SPARQL queries against your OML model to generate the JSON data files in `../../build/results/`. See the SPARQL queries provided:
- `stakeholders.sparql`
- `requirements.sparql`
- `missions.sparql`
- `capabilities.sparql`
- `capability-dependencies.sparql`

## Stakeholders
The Project Management Dashboard system serves two primary stakeholder groups: developers and project managers.

In [1]:
from utilities import *
df = dataframe("stakeholders.json")
df

,stakeholder_id,stakeholder_desc
0,developer,Developers working in each subproject who need...
1,project-manager,Project manager who oversees all subprojects a...


## Requirements
The system has three main requirements driven by stakeholder needs.

In [2]:
from utilities import *
df = dataframe("requirements.json")

# Get unique requirements (since one requirement can have multiple stakeholders)
requirements = df.drop_duplicates(subset=['req_id'])[['req_id', 'req_desc']]
requirements = requirements.rename(columns={"req_id": "Requirement ID", "req_desc": "Description"})

style = requirements.style.hide(axis="index").set_properties(**{'text-align': 'left', 'font-size': '11pt'})
style.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

Requirement ID,Description
r1-visual-overview,The dashboard must provide a visual overview of the dependencies and current releases for each sub-project.
r2-file-compatibility,The dashboard must be compatible with dependency specification files in each subproject's repository.
r3-up-to-date,The dashboard must have up-to-date versions and dependencies displayed in its graph.


## Missions
The dashboard delivers two key missions to satisfy user needs.

In [3]:
from utilities import *
df = dataframe("missions.json")
df = df.rename(columns={"mission_id": "Mission ID", "mission_desc": "Description"})

style = df.style.hide(axis="index").set_properties(**{'text-align': 'left', 'font-size': '11pt'})
style.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

Mission ID,Description
m1-at-a-glance-overview,"At a glance Overview: Needs clear visuals, displaying project connections and dependencies across the entire organization."
m2-easy-integration,Easy Integration: Needs the ability to access files in subproject repositories without complex setup or manual configuration.


## Capabilities
The system delivers three core capabilities that are derived from requirements.

In [4]:
from utilities import *
df = dataframe("capabilities.json")

# Get unique capabilities
capabilities = df.drop_duplicates(subset=['cap_id'])[['cap_id', 'cap_desc']]
capabilities = capabilities.rename(columns={"cap_id": "Capability ID", "cap_desc": "Description"})

style = capabilities.style.hide(axis="index").set_properties(**{'text-align': 'left', 'font-size': '11pt'})
style.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

Capability ID,Description
c1-authorization,Authorization: Server should gain authorization from GitHub App to access repository data.
c2-fetch-capability,Fetch Capability: Server should be able to fetch from each of the subproject repositories to grab their current version and dependencies.
c3-dashboard-display,Dashboard Display: Dashboard should be able to take fetched dependencies and display them in a graph format.
m1-at-a-glance-overview,"At a glance Overview: Needs clear visuals, displaying project connections and dependencies across the entire organization."
m2-easy-integration,Easy Integration: Needs the ability to access files in subproject repositories without complex setup or manual configuration.


## Capability Dependencies
The capabilities have dependencies on each other, forming a chain of functionality.

In [5]:
from utilities import *
df = dataframe("capability-dependencies.json")
capabilities1 = todict(df, 'cap1_id', 'cap1_desc')
capabilities2 = todict(df, 'cap2_id', 'cap2_desc')
dependencies = tolists(df, 'cap1_id', 'cap2_id')

# Build PlantUML
uml = "@startuml\n"

# Add all unique capabilities as objects
all_caps = union(capabilities1, capabilities2)
for key, value in all_caps.items():
    # Use underscores for alias (internal), hyphens in label (display)
    alias = key.replace('-', '_')
    uml += 'object "'+key+'" as '+alias+' <<capability>>\n'

# Add dependencies using aliases with underscores
for row in dependencies:
    from_alias = row[0].replace('-', '_')
    to_alias = row[1].replace('-', '_')
    uml += from_alias+' --> '+to_alias+'\n'

uml += "@enduml\n"

diagram(uml)

![Alt text](http://www.plantuml.com/plantuml/img/TP313eCW44JlV0NnlWVrQeX_GfQ51Xeg1jP3-_LDhP5OxN5dTbucCnR6pCiZYcJkZbWsrC7DCNaWdD646FZPI2oIEhtgkkfo6EgXL4NqOB5uap1RiA7C4JT6htT3RyPVI0kui4yvl913chw0LX_4t_1LIG1roedB9kldcI16DzdFH6y=)

## Requirements to Capabilities Traceability
This diagram shows how capabilities are derived from requirements.

In [6]:
from utilities import *
df = dataframe("capabilities.json")

# Get unique capabilities (filter out missions if they appear in this file)
cap_df = df[df['cap_id'].str.startswith('c', na=False)]
capabilities_dict = todict(cap_df, 'cap_id', 'cap_desc')

# Get unique requirements from the req_id column
# Extract unique requirement IDs where they exist
req_ids = cap_df[cap_df['req_id'].notna()]['req_id'].unique()
requirements_dict = {req_id: req_id for req_id in req_ids}

# Get derivations (capability -> requirement)
derivations = tolists(cap_df, 'cap_id', 'req_id')

# Build UML with different stereotypes
uml = "@startuml\n"

# Add requirements
for key in requirements_dict.keys():
    if pd.notna(key):
        alias = key.replace('-', '_')
        uml += 'object "'+key+'" as '+alias+' <<requirement>>\n'

# Add capabilities
for key, value in capabilities_dict.items():
    if pd.notna(value):
        alias = key.replace('-', '_')
        uml += 'object "'+key+'" as '+alias+' <<capability>>\n'

# Add derivation relationships
for row in derivations:
    if pd.notna(row[0]) and pd.notna(row[1]):
        cap_alias = row[0].replace('-', '_')
        req_alias = row[1].replace('-', '_')
        uml += cap_alias+' ..> '+req_alias+' : isDerivedFrom\n'

uml += "@enduml\n"

diagram(uml)

![Alt text](http://www.plantuml.com/plantuml/img/hPB1ReCm38RlVOgeTrS2RXD2FKnx3Siahl043KiSgktf9w0RevJRhj-lxvSTvnoLWgQkFVZwdQsQOyZX8Ys3zLrFAhMqefz7Gz647CS4Luafoy0VIG9tVDMgMdeAI3se1qVAirzWwb4zZcJVos2GcAW5Ft0OX6-pdE6CyGVlDjWCA6dZWtoHYhzElirm1KrPMkffNkAldCE5rJQmfBEVuyHnuRiL1JYAJUqfE70I-vRk7GN-1dWF7aisSvvErSxBlnY9hnnaOFSMV7TuDk8_sAkhRQndAqtQzbwFxfclRlnvtm==)